# ☕ Análise de Vendas de Café — Máquina Automática

## 1. Contextualização do Projeto

Consiste na análise de vendas de uma máquina automática de café transformando dados brutos de 3.636 transações em inteligência de negócio, mapeando padrões de consumo, mix de produtos e oportunidades de otimização.

**Período analisado:** 01/03/2024 a 23/03/2025  

**Localização:** Ucrânia (UTC +2)  

**Quantidade total de vendas:** 3.636 transações  

**Moeda:** Hryvnia Ucraniana (UAH)

---

## 2. Dicionário de Dados (após tratamento do dataset)

| Coluna           | Descrição                              |
|------------------|------------------------------------------|
| `date`           | Data da transação                        |
| `datetime`       | Data e hora completa                     |
| `hour`           | Horário (HH:MM)                          |
| `valor_hryvnia`  | Valor da venda (UAH)                     |
| `coffee_name`    | Nome do produto                          |
| `tipo_pagamento` | Cartão ou dinheiro                       |
| `dia_semana`     | Dia da semana                            |
| `mes_nome`       | Mês da transação                         |
| `ano`            | Ano                                      |
| `semana_ano`     | Número da semana                         |
| `dia_mes`        | Dia do mês                               |
| `periodo_dia`    | Período do dia (manhã, tarde ou noite)   |
| `feriado_nome`   | Nome do feriado, quando aplicável        |


In [16]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

In [17]:
# Estilo gráfico
plt.rcParams['figure.facecolor'] = '#F8F6F3'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 16
plt.style.use('seaborn-v0_8-darkgrid')

PALETA_CORES = {
    'principal': '#4E342E',
    'secundaria': '#795548',
    'positivo': '#2E7D32',
    'negativo': '#B71C1C',
    'neutro': '#D7CCC8',
    'ABC_A': '#70574F',
    'ABC_B': '#A37D6E',
    'ABC_C': '#CBB490',
    'coffee': ['#6F4E37', '#A0522D', '#8B7355', '#D2B48C', '#BC8F8F', '#4E3629', '#3E2723']
}

# 1. CARREGAMENTO


In [18]:
print("📂 ETAPA 1: CARREGAMENTO E VALIDAÇÃO")
print("-" * 80)

df_raw = pd.read_csv('index_1.csv')
print(f"✓ Dataset carregado: {len(df_raw):,} registros × {df_raw.shape[1]} colunas")

df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], errors='coerce')
df_raw['date'] = df_raw['datetime'].dt.normalize()
df_raw = df_raw[df_raw['date'] <= '2025-03-23'].copy()

print(f"✓ Registros válidos: {len(df_raw):,}")
print(f"✓ Nulos: {df_raw.isnull().sum().sum()} | Duplicatas: {df_raw.duplicated().sum()}")
print(f"✓ Período: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")

📂 ETAPA 1: CARREGAMENTO E VALIDAÇÃO
--------------------------------------------------------------------------------
✓ Dataset carregado: 3,636 registros × 6 colunas
✓ Registros válidos: 3,636
✓ Nulos: 89 | Duplicatas: 0
✓ Período: 2024-03-01 → 2025-03-23


# 2. ENGENHARIA DE FEATURES


In [19]:
print("\n⏰ ETAPA 2: ENGENHARIA DE FEATURES")
print("-" * 80)

df_raw['hour'] = df_raw['datetime'].dt.strftime('%H:%M')
df_raw['hora_numerica'] = df_raw['datetime'].dt.hour
df_raw['dia_semana_num'] = df_raw['datetime'].dt.dayofweek
df_raw['mes_num'] = df_raw['datetime'].dt.month
df_raw['ano'] = df_raw['datetime'].dt.year
df_raw['semana_ano'] = df_raw['datetime'].dt.isocalendar().week
df_raw['dia_mes'] = df_raw['datetime'].dt.day

dias_map = {0:'Segunda-feira',1:'Terça-feira',2:'Quarta-feira',3:'Quinta-feira',4:'Sexta-feira',5:'Sábado',6:'Domingo'}
meses_map = {1:'Janeiro',2:'Fevereiro',3:'Março',4:'Abril',5:'Maio',6:'Junho',7:'Julho',8:'Agosto',9:'Setembro',10:'Outubro',11:'Novembro',12:'Dezembro'}

df_raw['dia_semana'] = df_raw['dia_semana_num'].map(dias_map)
df_raw['mes_nome'] = df_raw['mes_num'].map(meses_map)
df_raw['mes_ano'] = df_raw['mes_nome'] + '/' + df_raw['ano'].astype(str)

df_raw['tipo_pagamento'] = df_raw['cash_type'].map({'card': 'Cartão', 'cash': 'Dinheiro'})
df_raw['valor_hryvnia'] = df_raw['money'].astype(str).str.replace(',', '.').astype(float)

feriados_dict = {
    "2024-03-08": "Dia Internacional da Mulher",
    "2024-05-09": "Dia da Memória",
    "2024-06-23": "Pentecostes Ortodoxo",
    "2024-06-24": "Pentecostes Ortodoxo",
    "2024-06-28": "Dia da Constituição",
    "2025-01-07": "Natal Ortodoxo",
    "2025-01-08": "Natal Ortodoxo",
    "2025-03-08": "Dia Internacional da Mulher"
}
feriados_dt = {pd.to_datetime(k).date(): v for k, v in feriados_dict.items()}
df_raw['feriado_nome'] = df_raw['date'].dt.date.map(lambda x: feriados_dt.get(x, '-'))

def classificar_periodo(h):
    if 6 <= h < 12: return 'Manhã (06h-11h)'
    elif 12 <= h < 18: return 'Tarde (12h-17h)'
    else: return 'Noite (18h ou mais)'

df_raw['periodo_dia'] = df_raw['hora_numerica'].apply(classificar_periodo)

print("✓ Features criadas")


⏰ ETAPA 2: ENGENHARIA DE FEATURES
--------------------------------------------------------------------------------
✓ Features criadas


As novas variáveis criadas aumentaram a granularidade temporal (`hora`, `dia`, `semana`, `mês` e `feriado`), além de categorização por turno e método de pagamento.

**Insight:** Essa estrutura favorece análises de sazonalidade e comportamento de consumo, permitindo segmentação por contexto (horário, feriado e canal de pagamento).


# 3. EXPORTAÇÃO DE DADOS TRATADOS

In [20]:
print("\n💾 ETAPA 3: EXPORTAÇÃO DE DADOS TRATADOS")
print("-" * 80)

colunas_finais = [
    'date','datetime','coffee_name','hour','hora_numerica','tipo_pagamento','valor_hryvnia',
    'dia_semana_num','dia_semana','mes_num','mes_nome','ano','semana_ano','dia_mes','mes_ano',
    'periodo_dia','feriado_nome'
]

for c in colunas_finais:
    if c not in df_raw.columns:
        df_raw[c] = pd.NA

df_tratado = df_raw[colunas_finais].copy()
del df_raw

# Exporta arquivos
try:
    df_tratado.to_excel('vendas_cafe_tratado.xlsx', index=False, sheet_name='Vendas', engine='openpyxl')
    print("✓ Excel exportado: vendas_cafe_tratado.xlsx")
except Exception as e:
    print(f"❌ Erro exportando Excel: {e}")

try:
    df_tratado.to_csv('vendas_cafe_tratado.csv', index=False, encoding='utf-8-sig')
    print("✓ CSV exportado: vendas_cafe_tratado.csv")
except Exception as e:
    print(f"❌ Erro exportando CSV: {e}")


💾 ETAPA 3: EXPORTAÇÃO DE DADOS TRATADOS
--------------------------------------------------------------------------------
✓ Excel exportado: vendas_cafe_tratado.xlsx
✓ CSV exportado: vendas_cafe_tratado.csv


# 4. KPIs GLOBAIS




In [21]:
print("\n📈 ETAPA 4: INDICADORES-CHAVE DE PERFORMANCE (KPIs)")
print("-" * 80)

receita_total = df_tratado['valor_hryvnia'].sum()
total_transacoes = len(df_tratado)
ticket_medio = df_tratado['valor_hryvnia'].mean()
dias_operacao = df_tratado['date'].nunique()
vendas_cartao = len(df_tratado[df_tratado['tipo_pagamento'] == 'Cartão'])
vendas_dinheiro = len(df_tratado[df_tratado['tipo_pagamento'] == 'Dinheiro'])
vendas_feriados = len(df_tratado[df_tratado['feriado_nome'] != '-'])

print(f"Período analisado      : {df_tratado['date'].min().date()} → {df_tratado['date'].max().date()}")
print(f"Dias com operação      : {dias_operacao}")
print(f"Transações totais      : {total_transacoes:,}")
print(f"Receita total          : {receita_total:,.2f} UAH")
print(f"Ticket médio           : {ticket_medio:.2f} UAH")
print(f"Ticket mínimo          : {df_tratado['valor_hryvnia'].min():.2f} UAH")
print(f"Ticket máximo          : {df_tratado['valor_hryvnia'].max():.2f} UAH")
print(f"Desvio padrão          : {df_tratado['valor_hryvnia'].std():.2f} UAH")
print(f"Vendas média/dia       : {total_transacoes / dias_operacao:.2f}")
print(f"Receita média/dia      : {receita_total / dias_operacao:.2f} UAH")
print(f"Cartão                 : {vendas_cartao:,} ({vendas_cartao/total_transacoes*100:.1f}%)")
print(f"Dinheiro               : {vendas_dinheiro:,} ({vendas_dinheiro/total_transacoes*100:.1f}%)")
print(f"Vendas em feriados     : {vendas_feriados:,} ({vendas_feriados/total_transacoes*100:.1f}%)")


📈 ETAPA 4: INDICADORES-CHAVE DE PERFORMANCE (KPIs)
--------------------------------------------------------------------------------
Período analisado      : 2024-03-01 → 2025-03-23
Dias com operação      : 381
Transações totais      : 3,636
Receita total          : 115,431.58 UAH
Ticket médio           : 31.75 UAH
Ticket mínimo          : 18.12 UAH
Ticket máximo          : 40.00 UAH
Desvio padrão          : 4.92 UAH
Vendas média/dia       : 9.54
Receita média/dia      : 302.97 UAH
Cartão                 : 3,547 (97.6%)
Dinheiro               : 89 (2.4%)
Vendas em feriados     : 48 (1.3%)


# 5. ANÁLISE POR MÉTODO DE PAGAMENTO

In [22]:
print("\n💳 ETAPA 5: ANÁLISE POR MÉTODO DE PAGAMENTO")
print("-" * 80)

for metodo in ['Cartão', 'Dinheiro']:
    filtro = df_tratado[df_tratado['tipo_pagamento'] == metodo]
    qtd = len(filtro)
    receita = filtro['valor_hryvnia'].sum()
    ticket = filtro['valor_hryvnia'].mean()
    perc_qtd = (qtd / total_transacoes) * 100
    perc_receita = (receita / receita_total) * 100

    print(f"\n{metodo}:")
    print(f"  • Transações  : {qtd:>5} ({perc_qtd:>5.1f}% do volume)")
    print(f"  • Receita     : {receita:>10,.2f} UAH ({perc_receita:>5.1f}% do total)")
    print(f"  • Ticket médio: {ticket:>10.2f} UAH")


💳 ETAPA 5: ANÁLISE POR MÉTODO DE PAGAMENTO
--------------------------------------------------------------------------------

Cartão:
  • Transações  :  3547 ( 97.6% do volume)
  • Receita     : 112,245.58 UAH ( 97.2% do total)
  • Ticket médio:      31.65 UAH

Dinheiro:
  • Transações  :    89 (  2.4% do volume)
  • Receita     :   3,186.00 UAH (  2.8% do total)
  • Ticket médio:      35.80 UAH


# 6. ANÁLISE — IMPACTO DOS FERIADOS NAS VENDAS

In [23]:
print("\n📅 IMPACTO DOS FERIADOS NAS VENDAS")
print("-" * 80)

# Cria a coluna que identifica se é feriado
df_tratado['eh_feriado'] = df_tratado['feriado_nome'] != '-'

# Agrupa os dados
analise_feriados = df_tratado.groupby('eh_feriado').agg(
    Transacoes=('valor_hryvnia', 'count'),
    Receita=('valor_hryvnia', 'sum'),
    Ticket_Medio=('valor_hryvnia', 'mean')
)

# Renomeia o índice para melhor leitura
analise_feriados.index = ['Dia normal', 'Feriado']

# Calcula o percentual da receita
analise_feriados['Perc_Receita_%'] = (
    analise_feriados['Receita'] / receita_total * 100
).round(2)

print(analise_feriados.to_string())


📅 IMPACTO DOS FERIADOS NAS VENDAS
--------------------------------------------------------------------------------
            Transacoes    Receita  Ticket_Medio  Perc_Receita_%
Dia normal        3588  113843.48     31.728952           98.62
Feriado             48    1588.10     33.085417            1.38


# 7. ANÁLISE ABC — MIX DE PRODUTOS

In [24]:
print("\n☕ ETAPA 7: ANÁLISE ABC — MIX DE PRODUTOS")
print("-" * 80)

abc = df_tratado.groupby('coffee_name').agg(
    Receita_Total=('valor_hryvnia', 'sum'),
    Quantidade=('valor_hryvnia', 'count'),
    Preco_Medio=('valor_hryvnia', 'mean')
).round(2).sort_values('Receita_Total', ascending=False)

abc['Receita_%'] = (abc['Receita_Total'] / receita_total * 100).round(2)
abc['Receita_Acumulada_%'] = abc['Receita_%'].cumsum().round(2)
abc['Classe_ABC'] = abc['Receita_Acumulada_%'].apply(
    lambda x: 'A' if x <= 80 else 'B' if x <= 95 else 'C'
)

print("\n" + abc.to_string())

print("\n📊 RESUMO ESTRATÉGICO ABC:")
for classe in ['A', 'B', 'C']:
    filtro = abc[abc['Classe_ABC'] == classe]
    print(f"\n  Classe {classe}: {len(filtro)} produtos | "
          f"{filtro['Receita_%'].sum():.1f}% receita | "
          f"{filtro['Quantidade'].sum():.0f} vendas")
    print(f"    → {', '.join(filtro.index.tolist())}")


☕ ETAPA 7: ANÁLISE ABC — MIX DE PRODUTOS
--------------------------------------------------------------------------------

                     Receita_Total  Quantidade  Preco_Medio  Receita_%  Receita_Acumulada_% Classe_ABC
coffee_name                                                                                           
Latte                     27866.30         782        35.63      24.14                24.14          A
Americano with Milk       25269.12         824        30.67      21.89                46.03          A
Cappuccino                18034.14         501        36.00      15.62                61.65          A
Americano                 15062.26         578        26.06      13.05                74.70          A
Hot Chocolate             10172.46         282        36.07       8.81                83.51          B
Cocoa                      8678.16         243        35.71       7.52                91.03          B
Cortado                    7534.86         292      

# 8. ANÁLISE MENSAL

In [25]:
print("\n📅 ETAPA 8: ANÁLISE TEMPORAL — PERFORMANCE MENSAL")
print("-" * 80)

mensal = df_tratado.groupby(['ano', 'mes_num', 'mes_ano']).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).reset_index().sort_values(['ano', 'mes_num']).reset_index(drop=True)

mensal['Perc_Total'] = (mensal['Receita'] / receita_total * 100).round(2)
mensal['Crescimento_%'] = mensal['Receita'].pct_change().mul(100).round(1)

# Flag para mês parcial
ultima_data = df_tratado['date'].max()
mensal['Parcial'] = mensal.apply(
    lambda r: (r['ano'] == ultima_data.year and r['mes_num'] == ultima_data.month and
               ultima_data.day < 28),
    axis=1
)

print("\n" + f"{'Período':<15} {'Receita':>12} {'Vendas':>8} {'Ticket':>10} "
      f"{'%Total':>8} {'Crescimento':>12}")
print("-" * 80)

for _, row in mensal.iterrows():
    parcial_flag = " (P)" if row['Parcial'] else ""
    print(f"{row['mes_ano']:<15} {row['Receita']:>12,.2f} {int(row['Vendas']):>8} "
          f"{row['Ticket_Medio']:>10.2f} {row['Perc_Total']:>7.1f}% "
          f"{str(row['Crescimento_%']):>11}%{parcial_flag}")


📅 ETAPA 8: ANÁLISE TEMPORAL — PERFORMANCE MENSAL
--------------------------------------------------------------------------------

Período              Receita   Vendas     Ticket   %Total  Crescimento
--------------------------------------------------------------------------------
Março/2024          7,050.20      206      34.22     6.1%         nan%
Abril/2024          6,720.56      196      34.29     5.8%        -4.7%
Maio/2024           9,063.42      267      33.95     7.8%        34.9%
Junho/2024          7,758.76      227      34.18     6.7%       -14.4%
Julho/2024          6,915.94      237      29.18     6.0%       -10.9%
Agosto/2024         7,613.84      272      27.99     6.6%        10.1%
Setembro/2024       9,988.64      344      29.04     8.7%        31.2%
Outubro/2024       13,891.16      426      32.61    12.0%        39.1%
Novembro/2024       8,590.54      259      33.17     7.4%       -38.2%
Dezembro/2024       8,237.74      259      31.81     7.1%        -4.1%
Janeir

# 9. ANÁLISE SEMANAL

In [26]:
print("\n📆 ETAPA 9: ANÁLISE SEMANAL — SAZONALIDADE POR DIA DA SEMANA")
print("-" * 80)

ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira',
              'Sexta-feira', 'Sábado', 'Domingo']

semanal = df_tratado.groupby(pd.Categorical(df_tratado['dia_semana'], categories=ordem_dias, ordered=True)).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).round(2)

print("\n" + f"{'Dia da Semana':<20} {'Vendas':>8} {'Receita':>15} {'Ticket':>12} {'%Total':>10}")
print("-" * 80)

for dia in ordem_dias:
    row = semanal.loc[dia]
    perc = (row['Receita'] / receita_total) * 100
    print(f"{dia:<20} {int(row['Vendas']):>8} {row['Receita']:>15,.2f} "
          f"{row['Ticket_Medio']:>12.2f} {perc:>9.1f}%")

melhor_dia = semanal['Receita'].idxmax()
pior_dia = semanal['Receita'].idxmin()
print(f"\n🏆 Melhor dia: {melhor_dia}")
print(f"⚠️  Pior dia:  {pior_dia}")


📆 ETAPA 9: ANÁLISE SEMANAL — SAZONALIDADE POR DIA DA SEMANA
--------------------------------------------------------------------------------

Dia da Semana          Vendas         Receita       Ticket     %Total
--------------------------------------------------------------------------------
Segunda-feira             561       17,925.10        31.95      15.5%
Terça-feira               585       18,637.38        31.86      16.1%
Quarta-feira              510       16,093.46        31.56      13.9%
Quinta-feira              520       16,477.40        31.69      14.3%
Sexta-feira               544       17,257.66        31.72      15.0%
Sábado                    482       15,182.52        31.50      13.2%
Domingo                   434       13,858.06        31.93      12.0%

🏆 Melhor dia: Terça-feira
⚠️  Pior dia:  Domingo


# 10. ANÁLISE HORÁRIA

In [27]:
print("\n🕐 ETAPA 10: ANÁLISE HORÁRIA — PERÍODOS DO DIA")
print("-" * 80)

ordem_periodos = ['Manhã (06h-11h)', 'Tarde (12h-17h)', 'Noite (18h ou mais)']

por_periodo = df_tratado.groupby(pd.Categorical(df_tratado['periodo_dia'], categories=ordem_periodos, ordered=True)).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).round(2)

print("\n" + f"{'Período':<20} {'Vendas':>8} {'%Vendas':>10} {'Receita':>15} "
      f"{'%Receita':>10} {'Ticket':>12}")
print("-" * 80)

for periodo in ordem_periodos:
    row = por_periodo.loc[periodo]
    perc_vendas = (row['Vendas'] / total_transacoes) * 100
    perc_receita = (row['Receita'] / receita_total) * 100
    print(f"{periodo:<20} {int(row['Vendas']):>8} {perc_vendas:>9.1f}% "
          f"{row['Receita']:>15,.2f} {perc_receita:>9.1f}% {row['Ticket_Medio']:>12.2f}")

# Top 10 horários
print("\n🔥 TOP 10 HORÁRIOS DE PICO (por volume):")
print("-" * 80)

por_hora = df_tratado.groupby('hora_numerica').agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).round(2).sort_values('Vendas', ascending=False).head(10)

print(f"\n{'Horário':>8} {'Vendas':>8} {'%Total':>10} {'Receita':>15} {'Ticket':>12}")
print("-" * 80)

for hora, row in por_hora.iterrows():
    perc = (row['Vendas'] / total_transacoes) * 100
    print(f"{int(hora):02d}h:00  {int(row['Vendas']):>8} {perc:>9.1f}% "
          f"{row['Receita']:>15,.2f} {row['Ticket_Medio']:>12.2f}")


🕐 ETAPA 10: ANÁLISE HORÁRIA — PERÍODOS DO DIA
--------------------------------------------------------------------------------

Período                Vendas    %Vendas         Receita   %Receita       Ticket
--------------------------------------------------------------------------------
Manhã (06h-11h)          1216      33.4%       37,230.80      32.3%        30.62
Tarde (12h-17h)          1476      40.6%       46,978.80      40.7%        31.83
Noite (18h ou mais)      944      26.0%       31,221.98      27.0%        33.07

🔥 TOP 10 HORÁRIOS DE PICO (por volume):
--------------------------------------------------------------------------------

 Horário   Vendas     %Total         Receita       Ticket
--------------------------------------------------------------------------------
10h:00       349       9.6%       10,994.52        31.50
11h:00       294       8.1%        8,849.10        30.10
16h:00       283       7.8%        9,221.60        32.59
12h:00       249       6.8%       

In [28]:
diario = (df_tratado.groupby('date')
          .agg(vendas=('valor_hryvnia', 'size'), receita=('valor_hryvnia', 'sum'))
          .reset_index())
diario['feriado'] = diario['date'].dt.date.isin(list(feriados_dt))
print(diario.groupby('feriado')[['vendas', 'receita']].agg(['count', 'mean']).round(2))

diario['dow'] = diario['date'].dt.dayofweek
print(diario.groupby('dow')['receita'].agg(['count', 'mean']).round(1))

diario['mes'] = diario['date'].dt.to_period('M')
m = diario.groupby('mes').agg(dias=('date', 'nunique'), receita=('receita', 'sum'))
print((m['receita'] / m['dias']).round(1).tail(2))

print('Vendas antes das 07h:', (df_tratado['hora_numerica'] < 7).sum())

        vendas       receita        
         count  mean   count    mean
feriado                             
False      373  9.62     373  305.21
True         8  6.00       8  198.51
     count   mean
dow              
0       55  325.9
1       55  338.9
2       52  309.5
3       54  305.1
4       56  308.2
5       55  276.0
6       54  256.6
mes
2025-02    472.0
2025-03    434.2
Freq: M, dtype: float64
Vendas antes das 07h: 5


## 🔎 Etapa 11: Auditoria complementar (preços e normalização por dia)

In [29]:
print("\n🔎 ETAPA 11: AUDITORIA COMPLEMENTAR")
print("-" * 80)

# 1) Regimes de preço no cartão (sustenta a nota sobre variação de preços)
print("\n💲 Preços unitários por produto (cartão):")
precos = (df_tratado[df_tratado['tipo_pagamento'] == 'Cartão']
          .groupby(['coffee_name', 'valor_hryvnia'])['datetime']
          .agg(inicio='min', fim='max', n='count'))
print(precos.to_string())

# 2) Feriados: média por dia de operação
diario = (df_tratado.groupby('date')
          .agg(vendas=('valor_hryvnia', 'size'), receita=('valor_hryvnia', 'sum'))
          .reset_index())
diario['eh_feriado'] = diario['date'].dt.date.isin(list(feriados_dt))
print("\n🎉 Feriados × dias normais (por dia de operação):")
print(diario.groupby('eh_feriado')[['vendas', 'receita']].agg(['count', 'mean']).round(2))

# 3) Dia da semana: receita média por dia operado
diario['dow'] = diario['date'].dt.dayofweek
print("\n📆 Receita média por dia operado (0=segunda ... 6=domingo):")
print(diario.groupby('dow')['receita'].agg(['count', 'mean']).round(1))

# 4) Mensal: receita por dia operado (neutraliza o mês parcial)
diario['mes'] = diario['date'].dt.to_period('M')
m = diario.groupby('mes').agg(dias=('date', 'nunique'), receita=('receita', 'sum'))
m['receita_por_dia'] = (m['receita'] / m['dias']).round(1)
print("\n📅 Receita por dia operado, por mês:")
print(m)

# 5) Período do dia: vendas antes das 07h
print("\n🕕 Vendas antes das 07h:", (df_tratado['hora_numerica'] < 7).sum())


🔎 ETAPA 11: AUDITORIA COMPLEMENTAR
--------------------------------------------------------------------------------

💲 Preços unitários por produto (cartão):
                                                   inicio                     fim    n
coffee_name         valor_hryvnia                                                     
Americano           23.02         2024-07-08 12:02:50.421 2024-09-27 13:28:33.107   98
                    25.96         2024-09-29 10:48:45.050 2025-03-23 15:47:28.723  341
                    27.92         2024-04-21 14:15:37.541 2024-07-05 22:18:56.945   72
                    28.90         2024-03-01 13:46:33.006 2024-04-20 13:08:37.378   53
Americano with Milk 27.92         2024-07-06 19:10:39.580 2024-09-28 09:31:14.759  222
                    30.86         2024-09-28 17:26:16.704 2025-03-22 19:16:52.727  389
                    32.82         2024-04-20 15:22:40.717 2024-07-05 22:21:09.457  147
                    33.80         2024-03-01 15:39:47.726 